# Segmentación de clientes con perfil de negocio

> Este cuaderno **no forma parte del curso original**. Se añade para cubrir el entregable *"perfil de segmentos con recomendaciones de negocio"*, que el ejercicio de las semillas de trigo no permite abordar de forma realista.

El caso de uso que el propio curso menciona como aplicación típica del clustering: *"una organización de marketing puede querer separar a sus clientes en segmentos distintos y luego investigar cómo se comporta cada uno"*.

Aquí lo hacemos de punta a punta:

1. Explorar los datos de clientes.
2. Determinar el número óptimo de segmentos (**método del codo** + **silueta**).
3. Segmentar con **K-Means**.
4. **Perfilar** cada segmento y traducirlo a recomendaciones de negocio.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

clientes = pd.read_csv('../datasets/customers.csv')
print(f'{len(clientes)} clientes')
clientes.head()

In [ ]:
clientes[['AverageSpend','AverageFrequency']].describe().round(2)

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(clientes['AverageSpend'], clientes['AverageFrequency'], alpha=0.7)
plt.xlabel('Gasto promedio por compra')
plt.ylabel('Frecuencia promedio de compra')
plt.title('Clientes (sin segmentar)')
plt.grid(alpha=0.3)
plt.show()

## Determinar el número de segmentos

Dos criterios a la vez: el **método del codo** (WCSS) y el **coeficiente de silueta**.

Ambas features se escalan antes, porque K-Means se basa en distancias y gasto y frecuencia están en unidades distintas.

In [ ]:
X = clientes[['AverageSpend','AverageFrequency']]
escalador = StandardScaler()
X_esc = escalador.fit_transform(X)

ks = range(2, 9)
wcss, siluetas = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0)
    etiquetas = km.fit_predict(X_esc)
    wcss.append(km.inertia_)
    siluetas.append(silhouette_score(X_esc, etiquetas))
    print(f'k = {k}  →  WCSS = {km.inertia_:7.2f}   silueta = {siluetas[-1]:.3f}')

k_optimo = ks[int(np.argmax(siluetas))]
print(f'\nMejor k según la silueta: {k_optimo}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,5))

ax[0].plot(list(ks), wcss, 'o-')
ax[0].set_xlabel('Número de clústeres (k)')
ax[0].set_ylabel('WCSS (inercia)')
ax[0].set_title('Método del codo')
ax[0].grid(alpha=0.3)

ax[1].plot(list(ks), siluetas, 'o-', color='green')
ax[1].axvline(k_optimo, color='red', linestyle='--', label=f'mejor k = {k_optimo}')
ax[1].set_xlabel('Número de clústeres (k)')
ax[1].set_ylabel('Coeficiente de silueta')
ax[1].set_title('Silueta')
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../imagenes/clientes-codo-y-silueta.png', dpi=110, bbox_inches='tight')
plt.show()

Los dos criterios coinciden, que es la situación ideal. El codo marca el punto donde añadir más segmentos deja de reducir la dispersión, y la silueta confirma que ahí la separación es máxima.

## Segmentar

In [ ]:
modelo = KMeans(n_clusters=k_optimo, n_init=10, random_state=0)
clientes['Segmento'] = modelo.fit_predict(X_esc)

# Devolver los centroides a las unidades originales para poder graficarlos
centroides = escalador.inverse_transform(modelo.cluster_centers_)

print(f'Silueta final: {silhouette_score(X_esc, clientes["Segmento"]):.3f}')
print()
print(clientes['Segmento'].value_counts().sort_index().to_string())

In [ ]:
plt.figure(figsize=(9,7))
for s in sorted(clientes['Segmento'].unique()):
    sub = clientes[clientes['Segmento'] == s]
    plt.scatter(sub['AverageSpend'], sub['AverageFrequency'], s=90, alpha=0.8, label=f'Segmento {s}')

plt.scatter(centroides[:,0], centroides[:,1], marker='X', s=280,
            c='red', edgecolors='black', linewidths=1.5, label='Centroides', zorder=5)
plt.xlabel('Gasto promedio por compra')
plt.ylabel('Frecuencia promedio de compra')
plt.title(f'Segmentación de clientes (k={k_optimo})')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('../imagenes/clientes-segmentos.png', dpi=110, bbox_inches='tight')
plt.show()

## Perfilar los segmentos

**Este es el paso que ningún algoritmo hace por ti.** K-Means devuelve números (0, 1, 2, 3) sin significado. Convertirlos en algo accionable es trabajo humano.

Primero, los números de cada segmento:

In [ ]:
perfil = clientes.groupby('Segmento').agg(
    Clientes=('Name', 'count'),
    Gasto_medio=('AverageSpend', 'mean'),
    Frecuencia_media=('AverageFrequency', 'mean'),
).round(2)

# Valor estimado por cliente = cuánto gasta × cuántas veces compra
perfil['Valor_estimado'] = (perfil['Gasto_medio'] * perfil['Frecuencia_media']).round(2)
perfil['% del total'] = (perfil['Clientes'] / len(clientes) * 100).round(1)

perfil.sort_values('Valor_estimado', ascending=False)

Y ahora la parte interpretativa: ponerle nombre a cada segmento según dónde cae en los dos ejes.

In [ ]:
mediana_gasto = clientes['AverageSpend'].median()
mediana_frec = clientes['AverageFrequency'].median()

def nombrar(fila):
    gasto_alto = fila['Gasto_medio'] >= mediana_gasto
    frec_alta = fila['Frecuencia_media'] >= mediana_frec
    if gasto_alto and frec_alta:   return 'Alto valor'
    if gasto_alto and not frec_alta: return 'Compra grande esporádica'
    if not gasto_alto and frec_alta: return 'Comprador frecuente pequeño'
    return 'Bajo compromiso'

perfil['Perfil'] = perfil.apply(nombrar, axis=1)

print(f'Medianas de referencia → gasto: {mediana_gasto:.2f} · frecuencia: {mediana_frec:.2f}\n')
perfil[['Clientes', '% del total', 'Gasto_medio', 'Frecuencia_media', 'Valor_estimado', 'Perfil']].sort_values('Valor_estimado', ascending=False)

## Recomendaciones de negocio

Cada perfil pide una acción comercial distinta. La lógica es siempre la misma: **mover al cliente en el eje donde está flojo**.

| Perfil | Situación | Acción recomendada | Por qué |
|---|---|---|---|
| **Alto valor** | Gasta mucho y compra seguido | Programa de fidelización, acceso anticipado, atención preferente | Es el segmento que más aporta; el riesgo aquí no es que gaste poco, es **perderlo** |
| **Compra grande esporádica** | Gasta mucho pero viene poco | Recordatorios, suscripción o recompra programada | El ticket ya es bueno: la palanca es la **frecuencia** |
| **Comprador frecuente pequeño** | Viene mucho pero gasta poco | Venta cruzada, paquetes, umbral de envío gratis | Ya tiene el hábito: la palanca es el **ticket medio** |
| **Bajo compromiso** | Gasta poco y viene poco | Campaña de reactivación de bajo coste | No justifica inversión alta; medir antes si vale la pena retenerlo |

> **Cuidado con la interpretación.** Los nombres salen de comparar cada segmento contra la mediana del propio dataset, así que son **relativos a esta cartera de clientes**, no absolutos. Con otra base de clientes, el mismo gasto podría ser "alto" o "bajo".

## Cerrar el círculo: de clustering a clasificación

Como se mencionaba en el cuaderno anterior, estos segmentos ya etiquetados sirven como punto de partida para un **clasificador**: con `Segmento` como label, se puede entrenar un modelo supervisado que asigne automáticamente un cliente **nuevo** a su segmento, sin reejecutar el clustering.

Eso convierte un análisis puntual en algo operativo.

In [ ]:
# Los datos ya están listos para eso: features + label generado
clientes[['Name', 'AverageSpend', 'AverageFrequency', 'Segmento']].head(10)